# Marching Tetrahedra

Marching tetrahedra splits every cell into six tetrahedra and extracts
the surface per tetrahedron. A tetrahedron has no ambiguous sign
configurations, so the mesh is watertight and consistent by
construction. The price is roughly 2-3x more triangles than marching
cubes, with worse triangle shapes and a slight directional grain from
the cell split. The method goes back to Doi and Koide (1991).


In [1]:
import isoext
from isoext import viewer
from isoext.sdf import SphereSDF


## Basic Usage

The interface matches the other extraction methods: a grid with scalar
values in, vertex and face tensors out.


In [2]:
grid = isoext.UniformGrid([32, 32, 32])
grid.set_values(SphereSDF(radius=0.7)(grid.get_points()))

vertices, faces = isoext.marching_tetrahedra(grid)
print(f"marching_tetrahedra: {faces.shape[0]:,} triangles")

v_mc, f_mc = isoext.marching_cubes(grid)
print(f"marching_cubes:      {f_mc.shape[0]:,} triangles")

viewer.embed(vertices, faces, color="tomato", flat_shading=True)


marching_tetrahedra: 13,416 triangles
marching_cubes:      4,508 triangles


## Tessellation

The wireframes on a coarse grid show where the extra triangles go and
the diagonal grain left by the six-tetrahedra split. Marching cubes
first, marching tetrahedra second.


In [3]:
coarse = isoext.UniformGrid([16, 16, 16])
coarse.set_values(SphereSDF(radius=0.7)(coarse.get_points()))

v, f = isoext.marching_cubes(coarse)
viewer.embed(v, f, color="steelblue", wireframe=True)


In [4]:
v, f = isoext.marching_tetrahedra(coarse)
viewer.embed(v, f, color="tomato", wireframe=True)


## When to Use It

Pick marching tetrahedra when guaranteed topology matters more than
triangle count, for example as input to solvers or simplification
passes that assume a closed manifold. For visual quality at the same
budget, marching cubes gives cleaner triangles.
